In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la
from periodic_simulation_setup import *

In [ ]:
Pressure = inflation.InflatablePeriodicUnit.EnergyType.Pressure
Elastic  = inflation.InflatablePeriodicUnit.EnergyType.Elastic
Full     = inflation.InflatablePeriodicUnit.EnergyType.Full

In [ ]:
n_vx = [[0, 0], [0, 1], [0, 2],
        [1, 0], [1, 1], [1, 2],
        [2, 0], [2, 1], [2, 2]]
n_edge = [(0, 1), (1, 2), 
          (3, 4), (4, 5),
          (6, 7), (7, 8),
          (0, 3), (3, 6),
          (1, 4), (4, 7),
          (2, 5), (5, 8)]
triArea = 0.5

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")

In [ ]:
fuseMarkers = [0] * 9

In [ ]:
# fuseMarkers[0] = 1
# fuseMarkers[2] = 1
# fuseMarkers[6] = 1
# fuseMarkers[8] = 1

In [ ]:
fuseMarkers

In [ ]:
np.array(fuseMarkers) == 1

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) == 0, epsilon = 1e-5)

In [ ]:
# ipu.set_use_planar_homogenization(False)

In [ ]:
import periodic_unit_helper

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
ipu.sheet.pressure = 10
ipu.sheet.setUseTensionFieldEnergy(False)
ipu.sheet.setUseHessianProjectedEnergy(False)
# ipu.sheet.disableFusedRegionTensionFieldTheory(False)

In [ ]:
ipu.getVars()

In [ ]:
ipu.numVars()

In [ ]:
vars = ipu.getVars()

In [ ]:
fluctuation = vars[3:-2].reshape(4, 3)

In [ ]:
fluctuation[:, 2] += 1

In [ ]:
vars[3:-2] = fluctuation.flatten()

In [ ]:
kappa = 0.5
alpha = 0
# ka_1 = kappa ** 0.5 * np.cos(alpha)
# ka_2 = kappa ** 0.5 * np.sin(alpha)
vars[-2] = kappa
vars[-1] = alpha

In [ ]:
ipu.setVars(vars)

In [ ]:
viewer.update()

In [ ]:
print("before", allEnergies(ipu), allGradientNorms(ipu))
print("inflatable sheet", np.linalg.norm(ipu.sheet.gradient()))
print("inflatable bending change vars first two ", np.linalg.norm(ipu.bent_sheet_gradient()[:-2]))
print("inflatable bending change vars", np.linalg.norm(ipu.bent_sheet_gradient()))



reparametrize_gamma_bar(ipu)
print("after ", allEnergies(ipu), allGradientNorms(ipu))
print("inflatable sheet", np.linalg.norm(ipu.sheet.gradient()))
print("inflatable bending change vars first two ", np.linalg.norm(ipu.bent_sheet_gradient()[:-2]))
print("inflatable bending change vars", np.linalg.norm(ipu.bent_sheet_gradient()))




In [ ]:
a = np.array([0, 1])

In [ ]:
a.reshape((2, 1)) * a.reshape((1, 2))

In [ ]:
ipu.getVars()

In [ ]:
np.linalg.norm(ipu.sheet.gradient())

In [ ]:
import fd_validation

In [ ]:
perturbation = np.random.random(ipu.bent_sheet_numVars()) * 1e-2

In [ ]:
class fd_wrapper:
    def __init__(self, ipu):
        self.ipu = ipu

    def setVars(self, v):
        self.ipu.bent_sheet_setVars(v)
    def numVars(self):
        return self.ipu.bent_sheet_numVars()

    def getVars(self):
        return self.ipu.bent_sheet_getVars()

    def energy(self):   return self.ipu.energy()
    def gradient(self): return self.ipu.bent_sheet_gradient()
    def hessian(self): return self.ipu.bent_sheet_hessian()

In [ ]:
fd_wrapper(ipu).numVars()

In [ ]:
ipu.numVars()

In [ ]:
fd_validation.gradConvergencePlot(fd_wrapper(ipu))

In [ ]:
ipu.get_kappa()

In [ ]:
fd_validation.hessConvergencePlot(fd_wrapper(ipu))

In [ ]:

v0_indices = [np.arange(0, ipu.sheet.numVars())]
v0_indices = np.array(v0_indices).flatten()

v1_indices = [np.arange(ipu.sheet.numVars(), ipu.sheet.numVars() + 1)]
v1_indices = np.array(v1_indices).flatten()


v0_star_indices = [np.arange(ipu.sheet.numVars() + 1, ipu.sheet.numVars() + 2)]
v0_star_indices = np.array(v0_star_indices).flatten()

In [ ]:

var_types = ['x', 'kappa', 'alpha']
var_indices = {'x': v0_indices,
               'kappa': v1_indices, 
               'alpha': v0_star_indices}


In [ ]:
periodic_unit_helper.getNumpyArrayFromCSC(ipu.bent_sheet_hessian())[:3, :3]

In [ ]:
fd_validation.hessian_convergence_block_plot(fd_wrapper(ipu), var_types, var_indices, testHessVec=False)

In [ ]:
ipu.numVars()

In [ ]:
ipu.setVars(ipu.getVars())

In [ ]:
fd_validation.gradConvergencePlot(ipu)

In [ ]:
fd_validation.hessConvergencePlot(ipu)

In [ ]:
fd_validation.gradConvergencePlot(ipu.sheet)

In [ ]:
fd_validation.hessConvergencePlot(ipu.sheet)

In [ ]:
ipu.sheet.numVars()

In [ ]:
H = ipu.bent_sheet_hessian()

In [ ]:
H = periodic_unit_helper.getNumpyArrayFromCSC(H)

In [ ]:
H.shape

In [ ]:
# H

In [ ]:
ipu.bent_sheet_gradient().shape

In [ ]:
ipu.sheet.gradient()

In [ ]:
# H

In [ ]:
sheet_hessian = periodic_unit_helper.getNumpyArrayFromCSC(ipu.sheet.hessian())

In [ ]:
bent_hessian = periodic_unit_helper.getNumpyArrayFromCSC(ipu.bent_sheet_hessian())

In [ ]:
np.linalg.norm(sheet_hessian[:27, :27] - bent_hessian[:27, :27])

In [ ]:
bent_hessian[-2:, -2:]

In [ ]:
fd_validation.hessConvergencePlot(ipu.sheet, testHessVec=False)